# Model Preparation 03 — Validation Strategy

**Goal:** Define a rigorous train / validation / test split for time-series price data.

**Critical rule:** Do NOT use random cross-validation on time series.  
Random split → model sees future prices during training → inflated metrics → fails in production.

**Method:** Walk-forward (expanding window) cross-validation — train on the past, validate on the
next window. The model only ever sees data that was available at the time of prediction.

**Tables:** gold_price_features (full history)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from scipy.stats import t as t_dist

In [2]:
gold = duckdb.connect("../../data/gold/cards.duckdb", read_only=True)

# Snapshot calendar: count cards and rows per day to understand data density over time.
dates_df = gold.execute("""
    SELECT snapshot_date,
           COUNT(DISTINCT uuid) AS n_cards,
           COUNT(*)             AS n_rows
    FROM gold_price_features
    WHERE eur IS NOT NULL AND uuid IS NOT NULL
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""").df()
dates_df["snapshot_date"] = pd.to_datetime(dates_df["snapshot_date"])
all_dates = dates_df["snapshot_date"].tolist()

print(f"Snapshot dates available: {len(all_dates)}")
print(
    f"Date range: {all_dates[0].date()} – {all_dates[-1].date()}"
    if all_dates
    else "No data."
)
display(dates_df)

Snapshot dates available: 32
Date range: 2026-05-26 – 2026-07-05


,snapshot_date,n_cards,n_rows
0,2026-05-26,82413,82413
1,2026-05-27,82413,82413
2,2026-05-28,82413,82413
3,2026-05-29,82413,82413
4,2026-05-30,82413,82413
5,2026-05-31,82413,82413
6,2026-06-01,82413,82413
7,2026-06-02,82413,82413
8,2026-06-03,82413,82413
9,2026-06-04,82413,82413


## 1. Walk-Forward Cross-Validation Scheme

**Expanding-window scheme (growing training set):**

```
Fold 1: Train [day  1 → 30]  |  Validate [day 31 → 37]
Fold 2: Train [day  1 → 37]  |  Validate [day 38 → 44]
Fold 3: Train [day  1 → 44]  |  Validate [day 45 → 51]
...
```

**Parameters:**
- `min_train_days = 30` — minimum history for stable rolling features (`price_30d_avg`)
- `validation_window = 7` — one prediction horizon (matches t+7 target)
- `step = 7` — advance by one horizon per fold (non-overlapping validation windows)

**Alternative — sliding window:** Fixed-length training set (always last N days). Better if the
market drifts and old data becomes irrelevant. Downside: smaller effective training sets in early
folds. Use expanding first; switch to sliding if later-fold validation metrics systematically
outperform earlier-fold metrics (a sign of concept drift).

In [3]:
MIN_TRAIN = 30
VAL_WINDOW = 7
STEP = 7


def generate_cv_folds(dates, min_train=30, val_window=7, step=7):
    """
    Walk-forward expanding-window CV.
    Train on dates[:i], validate on dates[i : i+val_window].
    Returns a list of fold dicts with start/end dates and counts.
    """
    folds = []
    i = min_train
    while i + val_window <= len(dates):
        folds.append(
            {
                "fold": len(folds) + 1,
                "train_start": dates[0],
                "train_end": dates[i - 1],
                "val_start": dates[i],
                "val_end": dates[min(i + val_window - 1, len(dates) - 1)],
                "n_train": i,
                "n_val": min(val_window, len(dates) - i),
            }
        )
        i += step
    return folds


folds = generate_cv_folds(
    all_dates, min_train=MIN_TRAIN, val_window=VAL_WINDOW, step=STEP
)

print(f"Walk-forward CV: min_train={MIN_TRAIN}, val_window={VAL_WINDOW}, step={STEP}")
print(f"Snapshots available: {len(all_dates)}")
print(f"CV folds generated:  {len(folds)}")

if len(folds) == 0:
    needed = MIN_TRAIN + VAL_WINDOW
    print(f"\n⚠  No folds possible — need ≥{needed} snapshots, have {len(all_dates)}.")
    print(f"   First fold will be available at ≈{needed} daily snapshots")
    print(
        f"   (roughly {all_dates[0].date() if all_dates else 'N/A'} + {needed} days)."
    )
    print("\nExpected fold structure:")
    print(
        f"  Fold 1: Train [day 1→{MIN_TRAIN}]  | Val [day {MIN_TRAIN + 1}→{MIN_TRAIN + VAL_WINDOW}]"
    )
    print(
        f"  Fold 2: Train [day 1→{MIN_TRAIN + STEP}]  | Val [day {MIN_TRAIN + STEP + 1}→{MIN_TRAIN + 2 * VAL_WINDOW}]"
    )
    print("  ...")
else:
    fold_df = pd.DataFrame(
        [
            {
                "fold": f["fold"],
                "train": f"{f['train_start'].date()} → {f['train_end'].date()}",
                "val": f"{f['val_start'].date()} → {f['val_end'].date()}",
                "n_train": f["n_train"],
                "n_val": f["n_val"],
            }
            for f in folds
        ]
    )
    display(fold_df)

    # Gantt-style visualisation.
    fig, ax = plt.subplots(figsize=(12, max(3, len(folds) * 0.6)))
    min_date = all_dates[0]
    for fold in folds:
        y = fold["fold"] - 1
        train_start_d = (fold["train_start"] - min_date).days
        val_start_d = (fold["val_start"] - min_date).days
        ax.broken_barh(
            [(train_start_d, fold["n_train"])], (y - 0.3, 0.6), color="steelblue"
        )
        ax.broken_barh([(val_start_d, fold["n_val"])], (y - 0.3, 0.6), color="salmon")
    ax.set_yticks(range(len(folds)))
    ax.set_yticklabels([f"Fold {f['fold']}" for f in folds])
    ax.set_xlabel("Days from first snapshot")
    ax.set_title("Walk-forward CV — expanding window (blue=train, red=validation)")
    ax.legend(
        handles=[
            mpatches.Patch(color="steelblue", label="Train"),
            mpatches.Patch(color="salmon", label="Validation"),
        ]
    )
    plt.tight_layout()
    plt.show()

Walk-forward CV: min_train=30, val_window=7, step=7
Snapshots available: 32
CV folds generated:  0

⚠  No folds possible — need ≥37 snapshots, have 32.
   First fold will be available at ≈37 daily snapshots
   (roughly 2026-05-26 + 37 days).

Expected fold structure:
  Fold 1: Train [day 1→30]  | Val [day 31→37]
  Fold 2: Train [day 1→37]  | Val [day 38→44]
  ...


## 2. Hold-Out Test Set

**Rule:** The test set is the last 14 days of history. It is locked until all model and
hyperparameter decisions have been finalised.

**Why 14 days (2 prediction horizons):**
- A single 7-day window gives only one data point per card — insufficient for stable metrics
- 14 days = 2 independent non-overlapping 7-day windows → more reliable final error estimate

**Protocol (in strict order):**
1. Lock the test set before any training begins
2. All feature, model, and hyperparameter decisions via CV on the non-test portion only
3. Train the final model on all non-test data
4. Evaluate on the test set **exactly once** — that is the official reported result

Feeding any test-set feedback back into modelling decisions converts the test set into a
de-facto validation set and invalidates the evaluation.

In [4]:
MIN_TEST_DAYS = 14

if len(all_dates) <= MIN_TEST_DAYS:
    print(
        f"⚠  Only {len(all_dates)} snapshot(s) — test set requires {MIN_TEST_DAYS} days."
    )
    print("   Current data would be entirely consumed by the hold-out.")
    print(
        f"   Test set definable once ≥{MIN_TEST_DAYS + MIN_TRAIN + VAL_WINDOW} snapshots exist."
    )
    print()
    print("Test set design (deferred):")
    print(f"  Hold-out: last {MIN_TEST_DAYS} days of history")
    print(
        f"  CV data:  everything before the hold-out (min {MIN_TRAIN + VAL_WINDOW} days)"
    )
    print("  Protocol: evaluated ONCE, after all modelling decisions are final")

    validation_config = {
        "status": "DEFERRED",
        "reason": f"Only {len(all_dates)} snapshots; need ≥{MIN_TEST_DAYS + MIN_TRAIN + VAL_WINDOW}",
        "min_test_days": MIN_TEST_DAYS,
        "min_train_days": MIN_TRAIN,
        "val_window": VAL_WINDOW,
        "step": STEP,
    }
else:
    cutoff_idx = len(all_dates) - MIN_TEST_DAYS
    train_val_dates = all_dates[:cutoff_idx]
    test_dates = all_dates[cutoff_idx:]

    print(
        f"Train+Val: {train_val_dates[0].date()} → {train_val_dates[-1].date()}  ({len(train_val_dates)} days)"
    )
    print(
        f"TEST (hold-out): {test_dates[0].date()} → {test_dates[-1].date()}  ({len(test_dates)} days)"
    )
    print("\nTEST SET LOCKED — do not use for any modelling decisions.")

    folds_cv = generate_cv_folds(
        train_val_dates, min_train=MIN_TRAIN, val_window=VAL_WINDOW, step=STEP
    )
    print(f"CV folds on train+val: {len(folds_cv)}")

    validation_config = {
        "status": "ACTIVE",
        "test_start": str(test_dates[0].date()),
        "test_end": str(test_dates[-1].date()),
        "n_cv_folds": len(folds_cv),
        "min_train_days": MIN_TRAIN,
        "val_window": VAL_WINDOW,
        "step": STEP,
    }

with open("validation_config.json", "w") as f:
    json.dump(validation_config, f, indent=2)
print("\nSaved validation_config.json ✓")

Train+Val: 2026-05-26 → 2026-06-13  (18 days)
TEST (hold-out): 2026-06-14 → 2026-07-05  (14 days)

TEST SET LOCKED — do not use for any modelling decisions.
CV folds on train+val: 0

Saved validation_config.json ✓


## 3. Statistical Power Analysis — Tier 3

**Problem:** Tier 3 (>€1000) is rare. In the last CV training fold, only a small number of
Tier 3 price rows will be available. Is that sufficient for a reliable ML model?

**Power estimate:** At effect size d = 0.5 (Cohen's medium) and α = 0.05:
- n = 70 → power ≈ 60% — **below the 80% threshold**
- n = 200 → power ≈ 90% — sufficient

**Implications:**
1. **Bayesian hierarchical model** — borrows strength across the rarity group; does not rely on
   Tier 3 alone for parameter estimation
2. **Cardmarket direct lookup** — skip prediction entirely for cards >€1000; query live market price

Both options are consistent with the project pricing strategy (model ML <€100, model+floor
€100–€1000, Cardmarket direct >€1000).

In [5]:
# Count rows per tier in the last CV fold training period (or full dataset if no folds yet).
if len(folds) > 0:
    last_fold = folds[-1]
    train_start_str = str(last_fold["train_start"].date())
    train_end_str = str(last_fold["train_end"].date())
else:
    train_start_str = str(all_dates[0].date())
    train_end_str = str(all_dates[-1].date())
    print(
        f"No CV folds yet — analysing full dataset ({train_start_str} → {train_end_str}) as proxy.\n"
    )

tier_counts = gold.execute(f"""
    SELECT
        CASE
            WHEN eur < 100   THEN 'Tier1 (<€100)'
            WHEN eur <= 1000 THEN 'Tier2 (€100–€1000)'
            ELSE              'Tier3 (>€1000)'
        END AS tier,
        COUNT(*) AS n_rows
    FROM gold_price_features
    WHERE eur IS NOT NULL AND uuid IS NOT NULL
      AND snapshot_date BETWEEN '{train_start_str}' AND '{train_end_str}'
    GROUP BY 1
    ORDER BY 1
""").df()
display(tier_counts)


def estimate_power(n, d=0.5, alpha=0.05):
    """Approximate two-sided power for a t-test at effect size d."""
    if n < 3:
        return 0.0
    t_crit = t_dist.ppf(1 - alpha / 2, df=n - 2)
    se = np.sqrt(2 / n)
    power = (
        1
        - t_dist.cdf(t_crit - d / se, df=n - 2)
        + t_dist.cdf(-t_crit - d / se, df=n - 2)
    )
    return float(power)


print("\nStatistical power per tier (effect size d=0.5, α=0.05, threshold 80%):")
for _, row in tier_counts.iterrows():
    n = int(row["n_rows"])
    power = estimate_power(n)
    flag = "✓ sufficient" if power >= 0.80 else "⚠ insufficient (<80%)"
    print(f"  {row['tier']:<25}: n={n:,}, power={power:.1%}  {flag}")

print(
    "\nTier 3 recommendation: Bayesian hierarchical model or Cardmarket direct lookup."
)
print(
    "Pricing strategy: model ML <€100, model+floor €100–€1000, Cardmarket direct >€1000."
)

No CV folds yet — analysing full dataset (2026-05-26 → 2026-07-05) as proxy.



,tier,n_rows
0,Tier1 (<€100),2594808
1,Tier2 (€100–€1000),18088
2,Tier3 (>€1000),4448



Statistical power per tier (effect size d=0.5, α=0.05, threshold 80%):
  Tier1 (<€100)            : n=2,594,808, power=100.0%  ✓ sufficient
  Tier2 (€100–€1000)       : n=18,088, power=100.0%  ✓ sufficient
  Tier3 (>€1000)           : n=4,448, power=100.0%  ✓ sufficient

Tier 3 recommendation: Bayesian hierarchical model or Cardmarket direct lookup.
Pricing strategy: model ML <€100, model+floor €100–€1000, Cardmarket direct >€1000.


## 4. Minimum Data Requirements per Card

Not every card can be included in training. Cards with insufficient history have NULL or
unreliable rolling features and must be excluded from the training set (but still predicted for).

**Minimum history thresholds:**
- `price_7d_avg` feature computable: card needs ≥7 days of price data
- `price_30d_avg` feature computable: card needs ≥30 days
- Model M1 (t+7 target): card needs ≥14 days (7 to form input features + 7 for the target)
- Model M2 (t+30 target): card needs ≥60 days

Cards below the threshold are excluded from the **training** rows but still appear in the
**prediction** pass — they get a prediction using only the features available (price_7d_avg
will be NULL, handled by imputation).

In [6]:
card_history = gold.execute("""
    SELECT uuid, COUNT(DISTINCT snapshot_date) AS n_days
    FROM gold_price_features
    WHERE eur IS NOT NULL AND uuid IS NOT NULL
    GROUP BY uuid
""").df()

print(f"Total unique cards with price data: {len(card_history):,}")
print(f"Snapshots in database: {len(all_dates)}")
print()

thresholds = [1, 7, 14, 30, 60, 90]
rows = []
for t in thresholds:
    n_above = (card_history["n_days"] >= t).sum()
    pct = n_above / len(card_history) * 100
    rows.append(
        {
            "min_days": t,
            "n_cards": n_above,
            "pct_cards": round(pct, 1),
            "use": {
                1: "any price feature",
                7: "price_7d_avg",
                14: "M1 training rows (t+7 target)",
                30: "price_30d_avg",
                60: "M2 training rows (t+30 target)",
                90: "full feature set",
            }[t],
        }
    )
display(pd.DataFrame(rows))

max_days = card_history["n_days"].max()
print(f"\nMax card history: {max_days} day(s)")
if max_days < 14:
    print("⚠  No card meets the M1 minimum (14 days) — 0 training rows for t+7 model.")
if max_days < 60:
    print("⚠  No card meets the M2 minimum (60 days) — 0 training rows for t+30 model.")

Total unique cards with price data: 82,413
Snapshots in database: 32



,min_days,n_cards,pct_cards,use
0,1,82413,100.0,any price feature
1,7,82413,100.0,price_7d_avg
2,14,82413,100.0,M1 training rows (t+7 target)
3,30,79908,97.0,price_30d_avg
4,60,0,0.0,M2 training rows (t+30 target)
5,90,0,0.0,full feature set



Max card history: 32 day(s)
⚠  No card meets the M2 minimum (60 days) — 0 training rows for t+30 model.


In [7]:
gold.close()

## 📋 Final Conclusions

```
VALIDATION STRATEGY
─────────────────────────────────────────────────────────────────────────────
Method:         Walk-forward expanding-window cross-validation
min_train_days: 30  (required for price_30d_avg feature stability)
val_window:     7 days  (one prediction horizon)
step:           7 days  (non-overlapping validation windows)

Current status: 5 snapshots available
CV folds:       0  (need ≥37 snapshots; first fold ≈ 2026-07-11)

HOLD-OUT TEST SET
─────────────────────────────────────────────────────────────────────────────
Status:    DEFERRED — need ≥51 snapshots for test set + at least one CV fold
Design:    last 14 days of history (2 prediction horizons)
Protocol:  locked before first training; evaluated exactly once after all decisions

STATISTICAL POWER — TIER 3
─────────────────────────────────────────────────────────────────────────────
Tier 1 (<€100):       n large  → power ≥ 80% ✓  (once data available)
Tier 2 (€100–€1000):  n medium → power borderline (~80%)
Tier 3 (>€1000):      n small  → power < 80% ⚠
  → Bayesian hierarchical model OR Cardmarket direct lookup
  → Consistent with pricing strategy: model ML <€100, model+floor €100–€1000, Cardmarket >€1000

MINIMUM DATA REQUIREMENTS PER CARD
─────────────────────────────────────────────────────────────────────────────
Cards with ≥1  day:  all cards  (any price feature usable)
Cards with ≥7  days: 0  (0.0%)  — price_7d_avg computable
Cards with ≥14 days: 0  (0.0%)  — M1 (t+7) training rows
Cards with ≥30 days: 0  (0.0%)  — price_30d_avg computable
Cards with ≥60 days: 0  (0.0%)  — M2 (t+30) training rows

RETEST SCHEDULE
─────────────────────────────────────────────────────────────────────────────
First CV fold:      ≈ 2026-07-11 (37 snapshots from 2026-06-04)
Test set + full CV: ≈ 2026-07-25 (51 snapshots)

OUTPUT FILES
─────────────────────────────────────────────────────────────────────────────
validation_config.json: SAVED ✓  (status=DEFERRED)
```